# TensorRT export and benchmark on Kaggle

Attach `best.pt` and the BDD100K YOLO validation set. Enable a Tesla T4 GPU accelerator.

In [ ]:
import subprocess
import sys
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("TensorRT export requires a CUDA GPU.")
print(f"GPU: {torch.cuda.get_device_name(0)}")

REPO_URL = "https://github.com/30nikamvaishnavi/autonomous-driving-perception-stack.git"
REPO_DIR = Path("/kaggle/working/autonomous-driving-perception-stack")
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml,export]"], cwd=REPO_DIR, check=True)
sys.path.insert(0, str(REPO_DIR / "src"))


In [ ]:
from ultralytics import YOLO

INPUT_DIR = Path("/kaggle/input")
BEST_PT = next(INPUT_DIR.rglob("best.pt"), None)
DATASET_YAML = next(INPUT_DIR.rglob("dataset.yaml"), None)
if BEST_PT is None:
    raise FileNotFoundError("No best.pt found under /kaggle/input.")
if DATASET_YAML is None:
    raise FileNotFoundError("No YOLO dataset.yaml found under /kaggle/input.")

print(f"Weights: {BEST_PT}")
print(f"YOLO val set: {DATASET_YAML}")

import tensorrt as trt
import ultralytics

print(f"TensorRT: {trt.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")


In [ ]:
import shutil

WORK_DIR = Path("/kaggle/working")
WORKING_MODEL = WORK_DIR / "best.pt"
shutil.copy2(BEST_PT, WORKING_MODEL)

ONNX_PATH = Path(YOLO(str(WORKING_MODEL)).export(format="onnx", imgsz=640, opset=17, simplify=True, dynamic=False))
print(f"ONNX: {ONNX_PATH} ({ONNX_PATH.stat().st_size / 1024**2:.1f} MB)")


In [ ]:
# Build the YOLO val set once and evaluate PyTorch on the same 2,000-image pool.
model = YOLO(str(WORKING_MODEL))
pytorch_metrics = model.val(data=str(DATASET_YAML), split="val", imgsz=640, batch=1, device=0)
print(f"PyTorch mAP50={pytorch_metrics.box.map50:.3f}, mAP50-95={pytorch_metrics.box.map:.3f}")


In [ ]:
ENGINE_DIR = WORK_DIR / "engines"
ENGINE_DIR.mkdir(parents=True, exist_ok=True)

model = YOLO(str(WORKING_MODEL))
fp16_export = Path(model.export(format="engine", device=0, imgsz=640, half=True))
FP16_ENGINE = ENGINE_DIR / "yolo_fp16.engine"
shutil.copy2(fp16_export, FP16_ENGINE)

model = YOLO(str(WORKING_MODEL))
int8_export = Path(model.export(
    format="engine", device=0, imgsz=640, int8=True,
    data=str(DATASET_YAML), fraction=300 / 2000, batch=1,
))
INT8_ENGINE = ENGINE_DIR / "yolo_int8.engine"
shutil.copy2(int8_export, INT8_ENGINE)

print(f"FP16 engine: {FP16_ENGINE}")
print(f"INT8 engine: {INT8_ENGINE}")


In [ ]:
fp16_metrics = YOLO(str(FP16_ENGINE), task="detect").val(
    data=str(DATASET_YAML), split="val", imgsz=640, batch=1, device=0
)
print(f"FP16 mAP50={fp16_metrics.box.map50:.3f}, mAP50-95={fp16_metrics.box.map:.3f}")

int8_metrics = YOLO(str(INT8_ENGINE), task="detect").val(
    data=str(DATASET_YAML), split="val", imgsz=640, batch=1, device=0
)
print(f"INT8 mAP50={int8_metrics.box.map50:.3f}, mAP50-95={int8_metrics.box.map:.3f}")


In [ ]:
import json
import time

import numpy as np

IMAGE = np.zeros((720, 1280, 3), dtype=np.uint8)
WARMUP = 30
ITERATIONS = 300

def benchmark_predict(model, label):
    for _ in range(WARMUP):
        model.predict(IMAGE, imgsz=[720, 1280], verbose=False)
    samples = []
    for _ in range(ITERATIONS):
        start = time.perf_counter()
        model.predict(IMAGE, imgsz=[720, 1280], verbose=False)
        samples.append((time.perf_counter() - start) * 1000.0)
    return {
        "backend": label,
        "mean_ms": float(np.mean(samples)),
        "p50_ms": float(np.percentile(samples, 50)),
        "p95_ms": float(np.percentile(samples, 95)),
        "p99_ms": float(np.percentile(samples, 99)),
        "fps": float(1000.0 / np.mean(samples)),
    }

results = [
    benchmark_predict(YOLO(str(WORKING_MODEL)), "PyTorch FP32 (CUDA)"),
    benchmark_predict(YOLO(str(FP16_ENGINE), task="detect"), "TensorRT FP16"),
    benchmark_predict(YOLO(str(INT8_ENGINE), task="detect"), "TensorRT INT8"),
]
baseline = results[0]["mean_ms"]
for row in results:
    row["speedup"] = baseline / row["mean_ms"]

metric_pairs = [
    (pytorch_metrics.box.map50, pytorch_metrics.box.map),
    (fp16_metrics.box.map50, fp16_metrics.box.map),
    (int8_metrics.box.map50, int8_metrics.box.map),
]
for row, (map50, map5095) in zip(results, metric_pairs):
    row["mAP50"] = float(map50)
    row["mAP50-95"] = float(map5095)

benchmark = {
    "hardware": "Kaggle Tesla T4",
    "model": "YOLO11n best.pt",
    "tensorrt_version": trt.__version__,
    "ultralytics_version": ultralytics.__version__,
    "method": "End-to-end Ultralytics predict() latency including pre/postprocess on a 1280x720 image, batch 1, 300 iterations after 30 warm-up iterations",
    "evaluation_images": 2000,
    "results": results,
}
(WORK_DIR / "benchmark.json").write_text(json.dumps(benchmark, indent=2))

lines = [
    "| Backend | mean ms | p50 | p95 | p99 | FPS | speed-up | mAP50 | mAP50-95 |",
    "|---|---:|---:|---:|---:|---:|---:|---:|---:|",
]
for row in results:
    lines.append(
        f"| {row['backend']} | {row['mean_ms']:.2f} | {row['p50_ms']:.2f} | "
        f"{row['p95_ms']:.2f} | {row['p99_ms']:.2f} | {row['fps']:.1f} | "
        f"{row['speedup']:.2f}x | {row['mAP50']:.3f} | {row['mAP50-95']:.3f} |"
    )
(WORK_DIR / "benchmark.md").write_text("# TensorRT benchmark on Kaggle Tesla T4\n\n" + "\n".join(lines))
print((WORK_DIR / "benchmark.md").read_text())
